# Additional End of week Exercise - week 4
Given a piece of self contained EVM smart contract, we attempt to identify any reentrancy vulnerability that it has.

## Description
The goal of this notebook is to test the capabilities of 3 LLM models at identifying the most common class of EVM smart contract vulnerability. The re-entrancy vulnerability.
All 3 models were able to identify the vulnerability but testing on gpt-40-mini resulted in the inability to identify the vulnerability

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [4]:
MODEL_GPT = "gpt-4o-mini"
MODEL_GPT_5 = "gpt-5"
MODEL_LLAMA = "llama3.2"

ai_models = [
    ("gpt-4o-mini", MODEL_GPT, "gpt-4o-mini"),
    ("gpt-5", MODEL_GPT_5, "gpt-5"),
    ("llama3.2", MODEL_LLAMA, "ollama"),
]

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openAI = OpenAI()
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

SYSTEM_PROMPT = """
You'll be given an excerpt of a self contained smart contract and your job is to identify any reentrancy vulnerability in it.
this contract might have.
You are to indicate which line number contains the vulnerability and explain how an attacker could exploit it.
"""

In [ ]:
def stream_response(history, user_message, model_label):
    config_from_label = {label: (model_id, backend) for label, model_id, backend in ai_models}
    model_id, backend = config_from_label.get(model_label, ai_models[0][1:])

    client = ollama if backend == "ollama" else openAI
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant or ""})
    messages.append({"role": "user", "content": user_message})

    stream = client.chat.completions.create(
        model=model_id,
        messages=messages,
        stream=True,
    )
    for chunk in stream:
        part = chunk.choices[0].delta.content or ""
        if part:
            yield part

In [7]:
def chat(message, history, model_choice):
    if not message or not message.strip():
        return
    full = ""
    for chunk in stream_response(history, message, model_choice):
        full += chunk
        yield full

In [ ]:
model_selector = gr.Dropdown(
    choices=[label for label, _, _ in ai_models],
    value=ai_models[0][0],
    label="Model",
)

ui = gr.ChatInterface(
    chat,
    additional_inputs=[model_selector],
    title="AI auditor",
    description="Provide the code you'd like to be audited below",
)
ui.launch(share=True)